<a href="https://colab.research.google.com/github/jigneshpandya86-lab/ALMWork/blob/main/FileFormatting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install python-docx


In [ ]:
import docx
import re
from google.colab import files

def find_real_style_name(doc, expected_name):
    expected_clean = expected_name.lower().replace(' ', '')
    for style in doc.styles:
        if style.name == expected_name: return style.name
    for style in doc.styles:
        word_style_clean = re.sub(r'^[\d\.\s]+', '', style.name).lower().replace(' ', '')
        if word_style_clean == expected_clean: return style.name
    return expected_name

def determine_style_smart(paragraph):
    text = paragraph.text.strip()
    if not text: return None

    # TIER 1: Keywords
    clean_text = text.replace(':', '').strip().upper()
    main_headings = [
        "OBJECTIVE", "SCOPE", "RESPONSIBILITIES / ACCOUNTABILITY",
        "PROCEDURE", "DEFINATION", "DEFINITION", "ANNEXURES",
        "REFERENCE", "ABBREVIATIONS", "HISTORY OF REVISION"
    ]
    if clean_text in main_headings: return 'SOPhead' # Mapped to exact case

    # TIER 2: XML List Level
    ilvl_xml = paragraph._element.xpath('./w:pPr/w:numPr/w:ilvl/@w:val')
    if ilvl_xml:
        level = int(ilvl_xml[0])
        if level == 0: return 'SOPhead'
        elif level == 1: return 'SOPSubHead'
        elif level == 2: return 'SOPSubHead2'  # FIXED: Mapped Level 3 to SOPSubHead2
        elif level >= 3: return 'SOPSubHead3'  # FIXED: Mapped Level 4+ to SOPSubHead3

    # TIER 3: Text Fallback
    if re.match(r'^\d+\.\d+\.\d+\.\d+\.?\s+', text): return 'SOPSubHead3'  # 1.1.1.1.
    elif re.match(r'^\d+\.\d+\.\d+\.?\s+', text): return 'SOPSubHead2'     # 1.1.1.
    elif re.match(r'^\d+\.\d+\.?\s+', text): return 'SOPSubHead'           # 1.1.
    elif re.match(r'^\d+\.?\s+', text): return 'SOPhead'                   # 1.

    return 'SOPPara'

def format_sop_document(input_file, output_file):
    print(f"  -> Processing: {input_file}")
    doc = docx.Document(input_file)

    for paragraph in doc.paragraphs:
        try:
            paragraph.style = 'Normal'
        except KeyError:
            pass

        for run in paragraph.runs:
            run.bold = run.italic = run.underline = None
            run.font.name = run.font.size = run.font.color.rgb = None

        target_style = determine_style_smart(paragraph)

        if target_style:
            real_style_name = find_real_style_name(doc, target_style)
            try:
                paragraph.style = real_style_name
            except KeyError:
                print(f"     [!] WARNING: Could not apply '{real_style_name}'.")

    doc.save(output_file)
    print(f"  -> Formatted successfully!\n")

def main():
    print("Click 'Choose Files' to upload your SOP:")
    uploaded_files = files.upload()

    if not uploaded_files: return

    print("\n" + "-" * 40)
    for file_name in uploaded_files.keys():
        if not file_name.endswith('.docx'): continue

        new_file_name = f"Formatted_{file_name}"
        format_sop_document(file_name, new_file_name)
        files.download(new_file_name)

    print("-" * 40)
    print("All tasks complete!")

main()